# 個別值與移動全距管制圖（I-MR Chart）

> **課程定位**：基本工具篇（3/6）  
> **前置課程**：[02_控制圖基礎_Xbar_R圖](./02_控制圖基礎_Xbar_R圖.ipynb)  
> **學習路徑**：01 概論 → 02 X-bar & R → **03 I-MR** → 04 P Chart → 05 製程能力 → 06 綜合案例

## 學習目標
- 理解 I-MR 圖的適用場景（無法分組取樣時）
- 掌握移動全距（Moving Range）的計算方法
- 能夠繪製並判讀 I-MR 圖
- 了解常態性假設對 I-MR 圖的影響

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 為什麼需要 I-MR 圖？

在某些製造場景中，**無法在同一時間取多個樣本**形成子組：

| 場景 | 說明 |
|------|------|
| 化學批次製程 | 每批只有一個 pH 值 / 濃度讀數 |
| 破壞性測試 | 測試會毀損產品（如拉力測試） |
| 高成本量測 | 每次量測費用昂貴（如 CT 掃描） |
| 連續製程 | 每個時間點僅一個讀數（如溫度） |

### X-bar & R vs. I-MR 比較

| 特性 | X-bar & R | I-MR |
|------|-----------|------|
| 子組大小 | n = 2~10 | n = 1 |
| 變異估計 | 子組內全距 R | 連續兩點的移動全距 MR |
| 對非常態敏感度 | 較低（中央極限定理） | **較高** |
| 偵測靈敏度 | 較高 | 較低 |

## 2. 公式推導

### 移動全距（Moving Range）

$$MR_i = |X_i - X_{i-1}|, \quad i = 2, 3, \ldots, n$$

### 中心線與管制界限

**I 圖（Individual Chart）：**
$$CL = \bar{X} = \frac{1}{n}\sum_{i=1}^{n} X_i$$

$$UCL = \bar{X} + \frac{3\overline{MR}}{d_2}$$

$$LCL = \bar{X} - \frac{3\overline{MR}}{d_2}$$

**MR 圖（Moving Range Chart）：**
$$CL = \overline{MR} = \frac{1}{n-1}\sum_{i=2}^{n} MR_i$$

$$UCL = D_4 \times \overline{MR}$$

$$LCL = D_3 \times \overline{MR} = 0$$

> **常數值**：對於移動全距（span = 2），$d_2 = 1.128$，$D_3 = 0$，$D_4 = 3.267$

In [ ]:
def plot_imr(values, title="I-MR 控制圖", check_normality=True):
    """
    繪製 I-MR（個別值-移動全距）控制圖
    
    Parameters:
    -----------
    values : array-like
        個別量測值序列
    title : str
        圖表標題
    check_normality : bool
        是否進行常態性檢驗
    
    Returns:
    --------
    dict : 包含統計量與管制界限
    """
    values = np.array(values, dtype=float)
    n = len(values)
    
    # 計算移動全距
    MR = np.abs(np.diff(values))
    
    # 統計量
    x_bar = values.mean()
    MR_bar = MR.mean()
    
    # 常數 (span=2)
    d2 = 1.128
    D3 = 0
    D4 = 3.267
    
    # I 圖管制界限
    sigma_hat = MR_bar / d2
    UCL_I = x_bar + 3 * sigma_hat
    LCL_I = x_bar - 3 * sigma_hat
    
    # MR 圖管制界限
    UCL_MR = D4 * MR_bar
    LCL_MR = D3 * MR_bar
    
    # 繪圖
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    
    x_axis = range(1, n + 1)
    
    # --- I 圖 ---
    ooc_I = (values > UCL_I) | (values < LCL_I)
    ax1.plot(x_axis, values, 'b-o', markersize=4, label='個別值', zorder=3)
    if ooc_I.any():
        ax1.scatter(np.where(ooc_I)[0] + 1, values[ooc_I],
                    color='red', s=80, zorder=5, label='超出管制界限', marker='s')
    ax1.axhline(y=x_bar, color='green', linewidth=2, label=f'CL = {x_bar:.4f}')
    ax1.axhline(y=UCL_I, color='red', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_I:.4f}')
    ax1.axhline(y=LCL_I, color='red', linestyle='--', linewidth=1.5, label=f'LCL = {LCL_I:.4f}')
    ax1.set_ylabel('個別值 (I)', fontsize=12)
    ax1.set_title('I 圖（Individual Chart）', fontsize=13)
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # --- MR 圖 ---
    mr_axis = range(2, n + 1)
    ooc_MR = MR > UCL_MR
    ax2.plot(mr_axis, MR, 'r-s', markersize=4, label='移動全距', zorder=3)
    if ooc_MR.any():
        ax2.scatter(np.where(ooc_MR)[0] + 2, MR[ooc_MR],
                    color='darkred', s=80, zorder=5, label='超出管制界限', marker='D')
    ax2.axhline(y=MR_bar, color='green', linewidth=2, label=f'CL = {MR_bar:.4f}')
    ax2.axhline(y=UCL_MR, color='red', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_MR:.4f}')
    ax2.set_xlabel('樣本編號', fontsize=12)
    ax2.set_ylabel('移動全距 (MR)', fontsize=12)
    ax2.set_title('MR 圖（Moving Range Chart）', fontsize=13)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 常態性檢驗
    if check_normality and n >= 8:
        stat, p_value = stats.shapiro(values)
        print(f"\n\U0001f4ca Shapiro-Wilk 常態性檢驗: W = {stat:.4f}, p-value = {p_value:.4f}")
        if p_value < 0.05:
            print("\u26a0\ufe0f  警告：數據可能不符合常態分配 (p < 0.05)，I-MR 圖的管制界限可能不準確。")
            print("   建議：考慮資料轉換（如 Box-Cox）或使用其他方法。")
        else:
            print("\u2705 數據未顯著偏離常態分配 (p \u2265 0.05)")
    
    return {
        'x_bar': x_bar, 'MR_bar': MR_bar, 'sigma_hat': sigma_hat,
        'UCL_I': UCL_I, 'LCL_I': LCL_I,
        'UCL_MR': UCL_MR, 'LCL_MR': LCL_MR,
        'ooc_I': np.where(ooc_I)[0] + 1,
        'ooc_MR': np.where(ooc_MR)[0] + 2,
        'MR': MR
    }

## 3. 實務案例一：化學蝕刻液 pH 值監控

### 情境描述

某半導體廠的化學蝕刻站，每 8 小時班次量測一次蝕刻液 pH 值。
- 目標 pH 值：**7.20**
- 收集 **40 個班次**的數據
- 在第 30 個班次後，蝕刻液老化導致 pH 值**逐漸漂移**

In [ ]:
np.random.seed(42)

# 前 30 個班次穩定
stable_pH = np.random.normal(loc=7.20, scale=0.08, size=30)

# 後 10 個班次逐漸漂移（每班次增加 0.03）
drift = np.array([7.20 + 0.03 * i for i in range(1, 11)])
drifting_pH = drift + np.random.normal(0, 0.08, size=10)

pH_data = np.concatenate([stable_pH, drifting_pH])

print("化學蝕刻液 pH 值監控數據")
print(f"總量測點數: {len(pH_data)}")
print(f"前 30 點: 穩定 (\u03bc=7.20, \u03c3=0.08)")
print(f"後 10 點: 蝕刻液老化導致漸進漂移\n")

result_ph = plot_imr(pH_data, title="化學蝕刻液 pH 值 I-MR 控制圖")

### 案例解讀

1. **I 圖**：後段數據點逐漸上升，部分超出 UCL → pH 值漂移
2. **MR 圖**：移動全距在後段可能略有增加，但主要問題在趨勢而非變異
3. **行動建議**：
   - 檢查蝕刻液使用壽命，安排更換
   - 建立 pH 值漂移的預警閾值
   - 考慮增加量測頻率以更早偵測漂移

## 4. 常態性假設的重要性

I-MR 圖的管制界限基於**常態分配假設**。與 X-bar & R 圖不同：
- X-bar 圖受**中央極限定理**保護——即使個別數據非常態，子組平均也趨近常態
- I-MR 圖**沒有**這個保護——每個點就是原始數據

### 常態性檢驗方法

| 方法 | 函數 | 說明 |
|------|------|------|
| Shapiro-Wilk | `scipy.stats.shapiro()` | 最常用，適合 n < 50 |
| Anderson-Darling | `scipy.stats.anderson()` | 對尾部敏感 |
| Q-Q Plot | 視覺化檢驗 | 直觀但主觀 |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 只用穩定段數據做常態性檢驗
stable_data = pH_data[:30]

# 直方圖 + 常態擬合
axes[0].hist(stable_data, bins=8, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x_fit = np.linspace(stable_data.min() - 0.1, stable_data.max() + 0.1, 100)
axes[0].plot(x_fit, stats.norm.pdf(x_fit, stable_data.mean(), stable_data.std()), 
             'r-', linewidth=2, label='常態擬合')
axes[0].set_title('直方圖 + 常態擬合', fontsize=13, fontweight='bold')
axes[0].set_xlabel('pH 值')
axes[0].set_ylabel('密度')
axes[0].legend()

# Q-Q Plot
stats.probplot(stable_data, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot（常態機率圖）', fontsize=13, fontweight='bold')
axes[1].get_lines()[0].set_markerfacecolor('steelblue')
axes[1].get_lines()[0].set_markersize(6)

# Box Plot
axes[2].boxplot(stable_data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[2].set_title('盒鬚圖', fontsize=13, fontweight='bold')
axes[2].set_ylabel('pH 值')

plt.tight_layout()
plt.show()

# 統計檢驗
stat_sw, p_sw = stats.shapiro(stable_data)
stat_ad = stats.anderson(stable_data, dist='norm')

print(f"Shapiro-Wilk 檢驗: W = {stat_sw:.4f}, p = {p_sw:.4f}")
print(f"Anderson-Darling 檢驗: 統計量 = {stat_ad.statistic:.4f}")
print(f"  臨界值 (5%): {stat_ad.critical_values[2]:.4f}", 
      "\u2192 通過 \u2705" if stat_ad.statistic < stat_ad.critical_values[2] else "\u2192 不通過 \u26a0\ufe0f")

## 5. 實務案例二：PCB 板厚量測

### 情境描述

某 PCB 製造廠量測成品板厚，每個生產批次取一片進行破壞性測試。
- 目標板厚：**1.60 mm**
- 收集 **50 個批次**的數據
- 第 35 批次後，壓合設備的壓力控制閥出現問題，造成**突然偏移**

In [ ]:
np.random.seed(88)

# 前 35 批正常
normal_pcb = np.random.normal(loc=1.60, scale=0.015, size=35)

# 後 15 批偏移
shifted_pcb = np.random.normal(loc=1.64, scale=0.015, size=15)

pcb_data = np.concatenate([normal_pcb, shifted_pcb])

print("PCB 板厚量測數據")
print(f"總量測點數: {len(pcb_data)}")
print(f"前 35 批: 穩定 (\u03bc=1.60mm, \u03c3=0.015mm)")
print(f"後 15 批: 壓力控制閥異常 (\u03bc=1.64mm)\n")

result_pcb = plot_imr(pcb_data, title="PCB 板厚 I-MR 控制圖")

print(f"\nI 圖超出管制界限: 樣本 {list(result_pcb['ooc_I'])}")
print(f"MR 圖超出管制界限: 樣本 {list(result_pcb['ooc_MR'])}")

### 案例解讀

1. **I 圖**：第 35 批後多點超出 UCL → 清楚的平均偏移訊號
2. **MR 圖**：第 35~36 點的 MR 值特別大 → 偏移發生的轉折點
3. **行動建議**：
   - 立即檢查壓合設備的壓力控制閥
   - 對第 35 批後的產品進行全數檢驗
   - 確認修復後重新計算管制界限

> **知識補給站** \U0001f4a1  
> MR 圖上的突然跳升往往指出**偏移發生的時間點**，這對找出根本原因非常有幫助。

## 6. 本章小結

| 項目 | 內容 |
|------|------|
| 適用場景 | 連續型數據，每次僅一個量測值 |
| 核心公式 | MR = |X\u1d62 - X\u1d62\u208b\u2081|；UCL/LCL = X\u0304 \u00b1 3(MR\u0304/d\u2082) |
| 關鍵常數 | d\u2082 = 1.128, D\u2084 = 3.267（span = 2） |
| 注意事項 | 對非常態數據敏感，需先做常態性檢驗 |
| 關鍵函數 | `plot_imr()` |

---

## 課堂練習

### \U0001f7e2 基礎題
1. 給定 10 個量測值：`[25.1, 25.3, 25.0, 25.2, 25.4, 25.1, 25.3, 25.2, 25.0, 25.3]`
   - 手動計算所有 MR 值
   - 計算 MR\u0304
   - 用程式驗證你的手算結果

### \U0001f7e1 進階題
2. 某製程量測數據的 Shapiro-Wilk p-value = 0.02。
   - 這代表什麼？
   - 你會如何處理？
   - 使用 `scipy.stats.boxcox()` 對數據進行 Box-Cox 轉換後，重新繪製 I-MR 圖。

### \U0001f534 挑戰題
3. 在 `plot_imr()` 函數中加入 **Western Electric Rules 檢查**（重用 Notebook 02 的函數邏輯），並在圖上用不同顏色標註不同規則的違規點。

---

> \U0001f4da **下一堂課**：[04_計數型管制圖_P_Chart](./04_計數型管制圖_P_Chart.ipynb) \u2014 處理「合格/不合格」的計數型數據